In [38]:
import warnings

sampling_rate = 256
duration      = 2.8
clip_length   = int(sampling_rate * duration)   # 716 samples
window_size = 10 # Length of window (samples)
step_size = 5 # Length of step (samples)

patients = [
    {
        "id": "patient_01",
        "normal_edf":    "chb01/chb01_01.edf",
        "pre_ictal_edf": "chb01/chb01_16.edf",
        "seizure_start_sec": 1015,   # patient-specific
    },
    {
        "id": "patient_02",
        "normal_edf":    "chb02/chb02_01.edf",
        "pre_ictal_edf": "chb02/chb02_16+.edf",
        "seizure_start_sec": 2972,
    },
    {
        "id": "patient_03",
        "normal_edf":    "chb03/chb03_06.edf",
        "pre_ictal_edf": "chb03/chb03_02.edf",
        "seizure_start_sec": 731,
    },
    {
        "id": "patient_04",
        "normal_edf":    "chb05/chb05_01.edf",
        "pre_ictal_edf": "chb05/chb05_13.edf",
        "seizure_start_sec": 1086,
    }
]

def build_patient_data(normal_windows, pre_ictal_windows):
    """Takes windowed data, applies STFT, returns X and y_labels."""
    
    # ── Combine windows ───────────────────────────────────────────
    all_normal = np.vstack(list(normal_windows.values()))
    X_windows  = np.vstack([all_normal, pre_ictal_windows])
    y_labels = np.hstack([
        np.zeros(len(all_normal)),
        np.ones(len(pre_ictal_windows))
    ]).astype(int)        

    # ── STFT each window ──────────────────────────────────────────
    stft_features = []
    for w in X_windows:
        _, _, Zxx = stft(w, nperseg=10)
        stft_features.append(np.log10(np.abs(Zxx) ** 2 + 1e-10))
    stft_features = np.array(stft_features)

    # ── Flatten ───────────────────────────────────────────────────
    X = stft_features.reshape(stft_features.shape[0], -1)
    
    return X, y_labels


def train_patient_model(X, y_labels):
    """Splits, scales, trains MLP, returns report."""
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
    )

    print(f"  y_train: {dict(zip(*np.unique(y_train, return_counts=True)))}")
    print(f"  y_test:  {dict(zip(*np.unique(y_test, return_counts=True)))}")
    print(f"  X shape: {X.shape}")

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        max_iter=2000,
        early_stopping=True,
        random_state=42
    )
    # compute weights based on class frequency
    from sklearn.utils.class_weight import compute_sample_weight
    sample_weights = compute_sample_weight(class_weight={0: 1, 1: 5}, y=y_train)
    
    mlp.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = mlp.predict(X_test)
    report = classification_report(y_test, y_pred, digits=4, output_dict=True)

    print(f"  Train accuracy : {mlp.score(X_train, y_train):.4f}")
    print(f"  Test accuracy  : {mlp.score(X_test, y_test):.4f}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=4))

    return report

all_reports = []

for patient in patients:
    print(f"\n{'='*40}")
    print(f"Patient: {patient['id']}")
    print(f"{'='*40}")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        raw       = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
        pre_ictal = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # ── Extract raw clips (your existing code) ────────────────────
    raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
    pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)

    # Normal clips
    total_samples = raw.n_times
    normal_clips_list = []
    for _ in range(10):
        start_sample = random.randint(0, total_samples - clip_length)
        chunk        = raw.get_data(start=start_sample, stop=start_sample + clip_length)
        normal_clips_list.append(chunk)
    X_normal = np.array(normal_clips_list)

    # Pre-ictal clip
    seizure_start    = patient["seizure_start_sec"]
    sph              = seizure_start - 600
    start_pre_ictal  = int(sph * sampling_rate)
    pre_ictal_data   = pre_ictal.get_data()
    pre_ictal_clip   = pre_ictal_data[0, start_pre_ictal:start_pre_ictal + clip_length]

    # ── Window (your existing code) ───────────────────────────────
    normal_windows = {}
    for i in range(10):
        normal_windows[f"clip_{i}"] = moving_window(X_normal[i, 0, :], window_size, step_size)
    pre_ictal_windows = moving_window(pre_ictal_clip, window_size, step_size)

    # ── Build dataset + train ─────────────────────────────────────
    X, y_labels  = build_patient_data(normal_windows, pre_ictal_windows)
    report       = train_patient_model(X, y_labels)
    all_reports.append({"id": patient["id"], "report": report})

# ── Mean across all patients ──────────────────────────────────────
print(f"\n{'='*40}")
print("MEAN ACROSS ALL PATIENTS")
print(f"{'='*40}")
for label, name in [("0", "Normal"), ("1", "Pre-ictal")]:
    avg_p = np.mean([r["report"][label]["precision"] for r in all_reports])
    avg_r = np.mean([r["report"][label]["recall"]    for r in all_reports])
    avg_f = np.mean([r["report"][label]["f1-score"]  for r in all_reports])
    print(f"  {name:10s} → precision: {avg_p:.4f}  recall: {avg_r:.4f}  f1: {avg_f:.4f}")
avg_acc = np.mean([r["report"]["accuracy"] for r in all_reports])
print(f"  Accuracy   → {avg_acc:.4f}")


Patient: patient_01


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:112: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:113: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.8959
  Test accuracy  : 0.9073
[[263  22]
 [  7  21]]
              precision    recall  f1-score   support

           0     0.9741    0.9228    0.9477       285
           1     0.4884    0.7500    0.5915        28

    accuracy                         0.9073       313
   macro avg     0.7312    0.8364    0.7696       313
weighted avg     0.9306    0.9073    0.9159       313


Patient: patient_02


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:112: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:113: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.9656
  Test accuracy  : 0.9681
[[275  10]
 [  0  28]]
              precision    recall  f1-score   support

           0     1.0000    0.9649    0.9821       285
           1     0.7368    1.0000    0.8485        28

    accuracy                         0.9681       313
   macro avg     0.8684    0.9825    0.9153       313
weighted avg     0.9765    0.9681    0.9702       313


Patient: patient_03


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:112: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:113: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.8991
  Test accuracy  : 0.8914
[[275  10]
 [ 24   4]]
              precision    recall  f1-score   support

           0     0.9197    0.9649    0.9418       285
           1     0.2857    0.1429    0.1905        28

    accuracy                         0.8914       313
   macro avg     0.6027    0.5539    0.5661       313
weighted avg     0.8630    0.8914    0.8746       313


Patient: patient_04


C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:112: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw          = mne.io.read_raw_edf(patient["normal_edf"],    preload=True, verbose=False)
C:\Users\Mary Yap\AppData\Local\Temp\ipykernel_40324\492931811.py:113: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  pre_ictal    = mne.io.read_raw_edf(patient["pre_ictal_edf"], preload=True, verbose=False)


  y_train: {np.int64(0): np.int64(1135), np.int64(1): np.int64(114)}
  y_test:  {np.int64(0): np.int64(285), np.int64(1): np.int64(28)}
  X shape: (1562, 18)
  Train accuracy : 0.8487
  Test accuracy  : 0.8435
[[236  49]
 [  0  28]]
              precision    recall  f1-score   support

           0     1.0000    0.8281    0.9060       285
           1     0.3636    1.0000    0.5333        28

    accuracy                         0.8435       313
   macro avg     0.6818    0.9140    0.7196       313
weighted avg     0.9431    0.8435    0.8726       313


MEAN ACROSS ALL PATIENTS
  Normal     → precision: 0.9735  recall: 0.9202  f1: 0.9444
  Pre-ictal  → precision: 0.4686  recall: 0.7232  f1: 0.5410
  Accuracy   → 0.9026
